In [1]:
from protocol_search import ProtocolSearcher

/home/tamerlan/Projects/arima_icd10_classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
searcher = ProtocolSearcher(
    chroma_persist_directory="./chroma_db",
    collection_name="medical_protocols",
    embedding_model_name="Qwen/Qwen3-Embedding-0.6B"
)

2026-02-22 01:14:58,954 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-22 01:14:58,956 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-02-22 01:14:59,027 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/config.json "HTTP/1.1 200 OK"
2026-02-22 01:14:59,224 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-22 01:14:59,295 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/tokenizer_config.json "HTTP/1.1 200 OK"
2026-02-22 01:14:59,497 - http

In [3]:
query="Здравствуйте. Пишу уже на нервах. Сыну 6 лет, год назад ставили шунт из‑за жидкости в голове. Долго всё было спокойно, а недели две как началось что‑то странное: утром просыпается с тяжёлой головой, жалуется, что «лоб давит», иногда его рвёт натощак. Становится вялый, потом наоборот капризный, плачет без причины. Говорит, что свет режет глаза, пару раз косил глазиком и как будто смотрит вниз «через ресницы». Когда идём, его качает, будто пол уплывает, пару раз чуть не упал. Вечером бывает лучше, а потом снова. Я очень боюсь, что опять что‑то не так с этой штукой. Температуры нет. Подскажите, это похоже на проблемы с оттоком? К кому бежать и как быстро? Не хочу тянуть, он же ребёнок…"

results = searcher.search(
    query=query,
    n_results=10,
    instruction="Given a search query, retrieve relevant passages that answer the query: "
)

2026-02-22 01:15:11,749 - protocol_search - INFO - Searching for: 'Здравствуйте. Пишу уже на нервах. Сыну 6 лет, год назад ставили шунт из‑за жидкости в голове. Долго всё было спокойно, а недели две как началось что‑то странное: утром просыпается с тяжёлой головой, жалуется, что «лоб давит», иногда его рвёт натощак. Становится вялый, потом наоборот капризный, плачет без причины. Говорит, что свет режет глаза, пару раз косил глазиком и как будто смотрит вниз «через ресницы». Когда идём, его качает, будто пол уплывает, пару раз чуть не упал. Вечером бывает лучше, а потом снова. Я очень боюсь, что опять что‑то не так с этой штукой. Температуры нет. Подскажите, это похоже на проблемы с оттоком? К кому бежать и как быстро? Не хочу тянуть, он же ребёнок…' (top 10 results)
2026-02-22 01:15:13,264 - protocol_search - INFO - Found 10 results


In [4]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- 5. SETUP CROSS-ENCODER (RERANKER) ---
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

reranker_model_name = 'Qwen/Qwen3-Reranker-0.6B'
print(f"Loading reranker model: {reranker_model_name} onto {device}...")

# Load Tokenizer
reranker_tokenizer = AutoTokenizer.from_pretrained(reranker_model_name)

# Load Model in float16 for 4GB VRAM safety.
# We use AutoModelForSequenceClassification with num_labels=1 for scoring.
cross_encoder = AutoModelForSequenceClassification.from_pretrained(
    reranker_model_name,
    num_labels=1,
    torch_dtype=torch.float16
).to(device)

# Set model to evaluation mode
cross_encoder.eval()

def rerank_results(query_text, initial_results, top_k=3):
    """
    Takes the initial results from ChromaDB and reranks them using a Qwen Cross-Encoder.
    Processes one pair at a time to prevent VRAM overflow.
    """
    
    if not initial_results or not initial_results['ids'] or not initial_results['ids'][0]:
        print("No results to rerank.")
        return []

    # Extract data from Chroma's dictionary output
    docs = initial_results['documents'][0]
    metadatas = initial_results['metadatas'][0]
    ids = initial_results['ids'][0]
    distances = initial_results['distances'][0]

    scores = []
    
    print("\nCalculating Cross-Encoder scores...")
    
    # 1. Process one Query-Document pair at a time (Batch Size = 1)
    with torch.no_grad():
        for doc in docs:
            # Tokenize pair. 
            # CRITICAL: We truncate to 2048 tokens. 4GB VRAM cannot handle infinite context.
            inputs = reranker_tokenizer(
                query_text,
                doc,
                padding=True,
                truncation=True,
                max_length=2048, 
                return_tensors="pt"
            ).to(device)
            
            # Predict
            output = cross_encoder(**inputs)
            
            # Extract the raw logit score
            score = float(output.logits.squeeze())
            scores.append(score)
            
            # Aggressive memory clearing for 4GB VRAM
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # 2. Sort by Cross-Encoder score in descending order
    scores = np.array(scores)
    ranked_indices = np.argsort(scores)[::-1]

    print(f"\n" + "="*60)
    print(f"RERANKED RESULTS FOR: '{query_text[:40]}...'")
    print("="*60)

    reranked_output = []
    
    # 3. Print and format the final top_k results
    for i, idx in enumerate(ranked_indices[:top_k]):
        doc_id = ids[idx]
        score = scores[idx]
        original_dist = distances[idx]
        metadata = metadatas[idx]
        document = docs[idx]
        
        print(f"Rank {i+1} (Was Rank {idx+1} before reranking)")
        print(f"Cross-Encoder Score: {score:.4f} | Original Dist: {original_dist:.4f}")
        print(f"Chunk ID: {doc_id} | Protocol: {metadata.get('protocol_id', 'N/A')}")
        print(f"Text Snippet: \n{document[:200]}...")
        print("-" * 60)
        
        reranked_output.append({
            "id": doc_id,
            "document": document,
            "metadata": metadata,
            "rerank_score": score,
            "original_distance": original_dist
        })
        
    return reranked_output


Loading reranker model: Qwen/Qwen3-Reranker-0.6B onto cuda...


2026-02-22 01:17:44,198 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Reranker-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-22 01:17:44,268 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Reranker-0.6B/6e9e69830b95c52b5fd889b7690dda3329508de3/config.json "HTTP/1.1 200 OK"
2026-02-22 01:17:44,465 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Reranker-0.6B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-22 01:17:44,534 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Reranker-0.6B/6e9e69830b95c52b5fd889b7690dda3329508de3/tokenizer_config.json "HTTP/1.1 200 OK"
2026-02-22 01:17:44,733 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-Reranker-0.6B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-02-22 01:17:44,935 - http

In [12]:
final_results = rerank_results(query, results, top_k=3)



Calculating Cross-Encoder scores...

RERANKED RESULTS FOR: 'Здравствуйте. Пишу уже на нервах. Сыну 6...'
Rank 1 (Was Rank 4 before reranking)
Cross-Encoder Score: 1.4033 | Original Dist: 0.1252
Chunk ID: p_de107231a8_chunk_2 | Protocol: p_de107231a8
Text Snippet: 
. ПРИЗНАКИ: Перелом скулоорбитального комплекса чаще всего встр ечается при ударе бокового отдела лица о твердые предметы, панели автосалона, при авто-травме; бетон, асфальт при падении с высоты, удар...
------------------------------------------------------------
Rank 2 (Was Rank 6 before reranking)
Cross-Encoder Score: 0.7061 | Original Dist: 0.1262
Chunk ID: p_8360af9a70_chunk_1 | Protocol: p_8360af9a70
Text Snippet: 
. ности словесными галлюцинациями и телесными Идеаторные, сенсопатические, кинестетические синдрома ощущениями. Галлюцинаторный вариант автоматизмы. Бредовый и псевдогаллюцинаторный Кандин- синдрома о...
------------------------------------------------------------
Rank 3 (Was Rank 1 before reranking)
Cross-E

In [8]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [6]:
import sqlite3
import json

DB_NAME = "protocols.db"
JSONL_FILE = "TaskQazCode/protocols_corpus.jsonl"

def setup_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Create table with protocol_id as the primary key for instant lookups
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS protocols (
            protocol_id TEXT PRIMARY KEY,
            source_file TEXT,
            title TEXT,
            data JSON
        )
    ''')
    
    # Insert data from JSONL
    with open(JSONL_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            cursor.execute('''
                INSERT OR REPLACE INTO protocols (protocol_id, source_file, title, data)
                VALUES (?, ?, ?, ?)
            ''', (
                obj['protocol_id'], 
                obj.get('source_file'), 
                obj.get('title'), 
                json.dumps(obj)
            ))
    
    conn.commit()
    conn.close()
    print("Database indexed successfully.")

def get_protocol_by_id(protocol_id):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute("SELECT data FROM protocols WHERE protocol_id = ?", (protocol_id,))
    row = cursor.fetchone()
    conn.close()
    
    return json.loads(row[0]) if row else None

# Run the setup once
setup_database()



Database indexed successfully.


In [10]:
# Fetch anytime
result = get_protocol_by_id("p_d57148b2d4")
print(result['text'] if result else "Not found")

Одобрен Объединенной комиссией по качеству медицинских услуг Министерства здравоохранения Республики Казахстан от «13» января 2023 года Протокол №177 КЛИНИЧЕСКИЙ ПРОТОКОЛ ДИАГНОСТИКИ И ЛЕЧЕНИЯ HELLP-СИНДРОМ I. ВВОДНАЯ ЧАСТЬ 1.1 Код(ы) МКБ-10: Код МКБ-10 O00-O99 Беременность, роды и послеродовой период О14.2 HELLP-синдром 1.2 Дата разработки/пересмотра протокола: 2022 год. 1.3 Сокращения, используемые в протоколе: аГУС – атипичный гемолитико-уремический синдром АЛТ – аланинаминотрансфераза АСТ – аспартатаминотрансфераза АЧТВ – активированное частичное тромбопластиновое время ВПХ – внутрипеченочный холестаз ДВС диссеминированного внутрисосудистого свертывания – синдром синдром КТГ – кардиотокография КТ – компьютерная томография ЛДГ – лактатдегидрогеназа НАЖБП – неалкогольная болезнь печени ОЖГБ – острый жировой гепатоз беременных ПВ – протромбиновое время ТТП – тромботическая тромбоцитопеническая пурпура УЗИ – ультразвуковое исследование 1.4 Пользователи протокола: акушеры-гинекологи, ан

In [13]:
final_results

[{'id': 'p_de107231a8_chunk_2',
  'document': '. ПРИЗНАКИ: Перелом скулоорбитального комплекса чаще всего встр ечается при ударе бокового отдела лица о твердые предметы, панели автосалона, при авто-травме; бетон, асфальт при падении с высоты, ударе по лицу. ЧМТ наблюдается в 40%. Помимо общих признаков переломов, характерно одностороннее нарушение зрения, кровоизлияние в конъюнктиву, кровотечение из носа, нарушение бокового движения нижней челюсти, затруднение или невозможность жевательных движений. Признаки при сочетанной закрытой черепно-мозговой травме: потеря памяти, головные боли, головокружение, тошнота иногда рвота - рвотные массы, кровь могут попасть, в дыхательные пути и вызвать аспирацию. Личная безопасность Осмотреть вокруг раненного, убедиться в безопасности НЕОТЛОЖНАЯ ПОМОЩЬ своих действии - взрывопасная ситуация, угроза обвала, наличия проводов электрической передачи, химических вредных веществ, опасных зверей и т.п. Проверить АВС (проходимость Устранить нарушение проходи

In [14]:
def get_ranked_protocols(reranked_chunks):
    """
    Aggregates chunk-level results into document-level (protocol) results
    using Max Pooling (highest scoring chunk dictates the document's score).
    """
    protocol_dict = {}

    for chunk in reranked_chunks:
        # Extract data safely
        protocol_id = chunk['metadata'].get('protocol_id', 'unknown')
        score = chunk['rerank_score']
        
        # If we haven't seen this protocol yet, OR if this current chunk 
        # has a higher score than the one we saved, update it.
        if protocol_id not in protocol_dict or score > protocol_dict[protocol_id]['score']:
            protocol_dict[protocol_id] = {
                'protocol_id': protocol_id,
                'score': score, # The cross-encoder score
                'title': chunk['metadata'].get('title', 'N/A'),
                'source_file': chunk['metadata'].get('source_file', 'N/A'),
                'icd_codes': chunk['metadata'].get('icd_codes_str', 'N/A'),
                'best_chunk_id': chunk['id'],
                'best_chunk_text': chunk['document']
            }

    # Convert the dictionary to a list and sort by score in descending order
    ranked_protocols = sorted(list(protocol_dict.values()), key=lambda x: x['score'], reverse=True)
    
    return ranked_protocols

# --- 7. AGGREGATE TO PROTOCOL LEVEL ---

# (Assuming 'final_results' is the list of dictionaries you provided)
final_protocols = get_ranked_protocols(final_results)

print("\n" + "="*60)
print("FINAL RANKED PROTOCOLS")
print("="*60)

for i, protocol in enumerate(final_protocols):
    print(f"Rank {i+1}: {protocol['protocol_id']}")
    print(f"Score: {protocol['score']:.4f}")
    print(f"File: {protocol['source_file']}")
    print(f"ICD Codes: {protocol['icd_codes']}")
    print(f"Matched based on chunk: {protocol['best_chunk_id']}")
    print("-" * 60)


FINAL RANKED PROTOCOLS
Rank 1: p_de107231a8
Score: 1.4033
File: Множественные (сочетанные) переломы лицевых костей и костей черепа. Сочетанная черепно-лицевая травма.pdf
ICD Codes: S02
Matched based on chunk: p_de107231a8_chunk_2
------------------------------------------------------------
Rank 2: p_8360af9a70
Score: 0.7061
File: Психические и поведенческие расстройства, вызванные употреблением алкоголя (для взрослых).pdf
ICD Codes: F10,F10.1,F10.2,F10.3,F10.4,F10.5
Matched based on chunk: p_8360af9a70_chunk_1
------------------------------------------------------------
Rank 3: p_a700bbe7d4
Score: 0.3772
File: Мигрень у взрослых.pdf
ICD Codes: G43,G43.0,G43.1,G43.2,G43.3,G43.8,G43.9
Matched based on chunk: p_a700bbe7d4_chunk_5
------------------------------------------------------------


In [16]:
import numpy as np

mock_reranked_results = [
    # --- PROTOCOL 1: Chronic Fatigue (2 chunks) ---
    {
        'id': 'p_fatigue_456_chunk_6',
        'document': '...астенический синдром, постоянное чувство усталости, не проходящее после отдыха, снижение работоспособности, туман в голове...',
        'metadata': {
            'protocol_id': 'p_fatigue_456',
            'total_chunks': 10,
            'chunk_number': 6,
            'icd_codes_str': 'G93.3',
            'source_file': 'Синдром хронической усталости.pdf',
            'title': 'Одобрено'
        },
        'rerank_score': np.float64(2.8541259765625), # Highest overall score
        'original_distance': 0.112345
    },
    {
        'id': 'p_fatigue_456_chunk_5',
        'document': '...дифференциальная диагностика проводится с эндокринными нарушениями, депрессией и пост-инфекционными состояниями...',
        'metadata': {
            'protocol_id': 'p_fatigue_456',
            'total_chunks': 10,
            'chunk_number': 5,
            'icd_codes_str': 'G93.3',
            'source_file': 'Синдром хронической усталости.pdf',
            'title': 'Одобрено'
        },
        'rerank_score': np.float64(0.4512332), # Lower score, should be ignored during Max Pooling
        'original_distance': 0.185566
    },

    # --- PROTOCOL 2: COVID Rehabilitation (3 chunks) ---
    {
        'id': 'p_covid_123_chunk_3',
        'document': '...у пациентов, перенесших новую коронавирусную инфекцию, часто наблюдается постковидный синдром: плаксивость, нарушение сна, раздражительность...',
        'metadata': {
            'protocol_id': 'p_covid_123',
            'total_chunks': 8,
            'chunk_number': 3,
            'icd_codes_str': 'U09.9',
            'source_file': 'Реабилитация после COVID-19.pdf',
            'title': 'Рекомендовано'
        },
        'rerank_score': np.float64(1.9876543), # Best score for the COVID protocol
        'original_distance': 0.145678
    },
    {
        'id': 'p_covid_123_chunk_1',
        'document': '...введение. Данный клинический протокол описывает методы реабилитации после вирусной пневмонии...',
        'metadata': {
            'protocol_id': 'p_covid_123',
            'total_chunks': 8,
            'chunk_number': 1,
            'icd_codes_str': 'U09.9',
            'source_file': 'Реабилитация после COVID-19.pdf',
            'title': 'Рекомендовано'
        },
        'rerank_score': np.float64(-0.543210), # Weak match
        'original_distance': 0.298765
    },
    {
        'id': 'p_covid_123_chunk_4',
        'document': '...рекомендуется консультация психолога или психотерапевта при сохранении подавленного настроения более 3 месяцев...',
        'metadata': {
            'protocol_id': 'p_covid_123',
            'total_chunks': 8,
            'chunk_number': 4,
            'icd_codes_str': 'U09.9',
            'source_file': 'Реабилитация после COVID-19.pdf',
            'title': 'Рекомендовано'
        },
        'rerank_score': np.float64(1.123456), # Good match, but chunk_3 is better
        'original_distance': 0.167890
    },

    # --- PROTOCOL 3: Depression (1 chunk) ---
    {
        'id': 'p_depression_789_chunk_2',
        'document': '...основные симптомы депрессивного эпизода: сниженное настроение, апатия, ангедония, нарушения сна (ранние пробуждения), изменение аппетита...',
        'metadata': {
            'protocol_id': 'p_depression_789',
            'total_chunks': 5,
            'chunk_number': 2,
            'icd_codes_str': 'F32.0',
            'source_file': 'Депрессивный эпизод.pdf',
            'title': 'Одобрено'
        },
        'rerank_score': np.float64(1.456789), # Only one chunk, so this is the score
        'original_distance': 0.134567
    }
]

final_protocols = get_ranked_protocols(mock_reranked_results)

for i, protocol in enumerate(final_protocols):
    print(f"Rank {i+1}: Protocol {protocol['protocol_id']} | Best Score: {protocol['score']:.4f}")

Rank 1: Protocol p_fatigue_456 | Best Score: 2.8541
Rank 2: Protocol p_covid_123 | Best Score: 1.9877
Rank 3: Protocol p_depression_789 | Best Score: 1.4568


In [17]:
final_protocols

[{'protocol_id': 'p_fatigue_456',
  'score': np.float64(2.8541259765625),
  'title': 'Одобрено',
  'source_file': 'Синдром хронической усталости.pdf',
  'icd_codes': 'G93.3',
  'best_chunk_id': 'p_fatigue_456_chunk_6',
  'best_chunk_text': '...астенический синдром, постоянное чувство усталости, не проходящее после отдыха, снижение работоспособности, туман в голове...'},
 {'protocol_id': 'p_covid_123',
  'score': np.float64(1.9876543),
  'title': 'Рекомендовано',
  'source_file': 'Реабилитация после COVID-19.pdf',
  'icd_codes': 'U09.9',
  'best_chunk_id': 'p_covid_123_chunk_3',
  'best_chunk_text': '...у пациентов, перенесших новую коронавирусную инфекцию, часто наблюдается постковидный синдром: плаксивость, нарушение сна, раздражительность...'},
 {'protocol_id': 'p_depression_789',
  'score': np.float64(1.456789),
  'title': 'Одобрено',
  'source_file': 'Депрессивный эпизод.pdf',
  'icd_codes': 'F32.0',
  'best_chunk_id': 'p_depression_789_chunk_2',
  'best_chunk_text': '...основные с